# WAXAL-NET fine-tuning on Colab

Works through `LANGUAGES` one at a time: train, evaluate, upload the results to the run's Hub repo.
Checkpoints go to the same `waxal-net-*` Hub repos as the Kaggle jobs, so a run started on either
platform resumes on the other. Give a language to one platform at a time (`jobs.py colab-claim`).

1. Runtime → Change runtime type → a GPU (L4 or A100 are much faster than T4).
2. Secrets (key icon, left bar) → add `HF_TOKEN` (a write token) and allow notebook access.
3. Run all. Keep the tab open: Colab Pro stops a session when its browser tab closes.
   If the session ends, run all again; training resumes from the last Hub checkpoint.

In [1]:
LANGUAGES = ['wolaytta', 'sidama', 'oromo', 'amharic']
MODEL = "mms_300m"
FLEURS_LANGUAGES = ['amharic', 'fula', 'lingala', 'luganda', 'oromo', 'shona']
TRAIN_HOURS = 22  # stop training before Colab's 24 h session limit

In [2]:
import os, subprocess, time
from pathlib import Path

SESSION_START = time.time()
REPO = Path("/content/asr_waxal")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jdaltonll02/asr_waxal.git", str(REPO)], check=True)

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.update(HF_HOME="/content/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
# The experiment configs write under /kaggle/working.
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
PY = "/tmp/waxal-venv/bin/python"
LOG_DIR = Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(*command, check=True):
    """Run a command from the repo root, streaming its output to the notebook and to LOG_DIR."""
    name = "_".join(Path(str(part)).stem for part in command[1:4])
    log_path = LOG_DIR / f"{time.strftime('%H%M%S')}_{name}.log"
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    with open(log_path, "w", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


run("bash", f"{REPO}/kaggle/setup_env.sh")

# The configs' per-device batch of 2 fits a 15 GB T4 on clips up to ~57 s; bigger
# GPUs take more per step (accumulation drops so the effective batch stays 32).
import torch
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = ["--per-device-batch-size", "8" if memory_gb > 35 else "4"] if memory_gb > 20 else []
print(f"{torch.cuda.get_device_name(0)}, {memory_gb:.0f} GB: {BATCH or 'config batch size'}")

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /tmp/waxal-venv
Activate with: source /tmp/waxal-venv/bin/activate
torch 2.8.0+cu128 | cuda True 1 | transformers 4.51.3 | datasets 3.6.0
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G  7.3G  229G   4% /
overlay         236G  7.3G  229G   4% /
NVIDIA L4, 22 GB: ['--per-device-batch-size', '4']


In [ ]:
import shutil
from huggingface_hub import HfApi

api = HfApi()
for LANGUAGE in LANGUAGES:
    print(f"===== {LANGUAGE} {MODEL} =====", flush=True)
    FLEURS = LANGUAGE in FLEURS_LANGUAGES
    RUN = f"finetune_{LANGUAGE}_{MODEL}"
    CONFIG = f"configs/experiment/{RUN}.yaml"
    CHECKPOINT = f"/kaggle/working/checkpoints/{RUN}/best_model"
    HUB = ["--hub-repo", f"waxal-net-{RUN}"] if os.environ.get("HF_TOKEN") else []
    run(PY, "-m", "src.train", "--config", CONFIG, "--deadline", str(SESSION_START + TRAIN_HOURS * 3600), *HUB, *BATCH)
    if Path(CHECKPOINT, "config.json").exists():
        run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT)
        if MODEL.startswith("whisper"):
            # Also score clips over 30 s in 30 s chunks; the paper doesn't say which it did.
            run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT, "--long-audio", "chunk30",
                "--output-dir", f"/kaggle/working/outputs/{RUN}_chunk30")
        if FLEURS:
            run(PY, "-m", "src.cross_domain_eval", "--config", f"configs/experiment/cross_domain_{LANGUAGE}_{MODEL}.yaml",
                "--checkpoint", CHECKPOINT)
    else:
        print("Training stopped at the session deadline. Run this notebook again to resume from the last checkpoint.")
    if not Path(CHECKPOINT, "config.json").exists():
        break  # stopped at the deadline; run all again to resume
    # Colab outputs are lost with the session; keep them next to the model.
    repo = f"{api.whoami()['name']}/waxal-net-{RUN}"
    for output in Path("/kaggle/working/outputs").glob(f"*_{LANGUAGE}_{MODEL}*"):
        api.upload_folder(repo_id=repo, folder_path=str(output), path_in_repo=f"outputs/{output.name}",
                          commit_message=f"{output.name} results")
    shutil.rmtree(f"/kaggle/working/checkpoints/{RUN}", ignore_errors=True)

===== wolaytta mms_300m =====

Fetching 1 files: 100%|██████████| 1/1 [00:01<00:00,  1.83s/it]
2026-11-17 14:51:50,392 INFO Restored /kaggle/working/checkpoints/finetune_wolaytta_mms_300m from hf.co/jdaltonII02/waxal-net-finetune_wolaytta_mms_300m
2026-11-17 14:57:19,602 INFO train: 41833 / 197.37 h -> 41831 / 197.37 h
2026-11-17 14:57:20,689 INFO validation: 2444 / 9.61 h -> 2444 / 9.61 h
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-11-17 14:57:33,112 INFO Effective batch 32 = 4 per device x 1 GPU(s) x 8 accumulation
2026-11-17 14:59:47,359 INFO progress {"step": 25, "max_steps": 39210, "epoch": 0.019124115509657678, "sec_per_step": null, "best_eval_loss": null, "loss": 17.6993, "grad_norm": 11.013168334960938, "learning_rate": 4.800000000000001e-06, "updat